# DEMO NHẬN DIỆN BIỂN SỐ XE VỚI YOLOV8 VÀ RAPIDOCR
Notebook này sử dụng mô hình YOLOv8 (`models/vision/best.onnx`) vừa huấn luyện từ Roboflow để định vị biển số xe, sau đó sử dụng RapidOCR để nhận diện ký tự biển số.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import onnxruntime as ort
from pathlib import Path

# Đảm bảo import được module từ src/
import sys
ROOT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

from src.vision.plate_detect import find_plate_regions
from src.vision.plate_ocr import recognize_plate

print("Tất cả thư viện đã sẵn sàng!")

### 1. Nạp ảnh kiểm thử và chạy mô hình YOLOv8

In [ ]:
# Chọn một ảnh xe mẫu trong thư mục data
image_path = ROOT_DIR / "data" / "processed" / "scans" / "20260920_224707_726477_biensoxe1.jpeg"
if not image_path.exists():
    image_path = ROOT_DIR / "data" / "external" / "plates" / "scene_59B112345.jpg"

img_bgr = cv2.imread(str(image_path))
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

# Chạy mô hình YOLOv8 vừa train để phát hiện vùng biển số
candidates = find_plate_regions(img_bgr)
print(f"Tìm thấy {len(candidates)} vùng ứng viên biển số.")

best_candidate = candidates[0]
bbox = best_candidate['bbox']
crop_bgr = best_candidate['crop']
print(f"Tọa độ biển số: {bbox} | Điểm tin cậy: {best_candidate['score']:.2f}")

### 2. Đọc ký tự biển số bằng OCR

In [ ]:
# Nhận diện ký tự trên ảnh crop biển số
ocr_result = recognize_plate(crop_bgr)

print(f"▶ Biển số nhận diện: {ocr_result.get('plate')}")
print(f"▶ Độ chính xác OCR : {ocr_result.get('confidence', 0)*100:.2f}%")
print(f"▶ Trạng thái hợp lệ: {'Hợp lệ' if ocr_result.get('valid') else 'Không hợp lệ'}")

### 3. Hiển thị hình ảnh trực quan

In [ ]:
# Vẽ bounding box lên ảnh gốc
annotated = img_rgb.copy()
x, y, w, h = bbox
plate_text = ocr_result.get('plate', '')

cv2.rectangle(annotated, (x, y), (x + w, y + h), (0, 255, 0), 3)
cv2.putText(annotated, f"{plate_text}", (x, max(30, y - 10)), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 2)

# Hiển thị bằng Matplotlib
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].imshow(annotated)
axes[0].set_title(f"YOLOv8 Detection: {plate_text}")
axes[0].axis('off')

axes[1].imshow(cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2RGB))
axes[1].set_title("Cropped Plate (Input to OCR)")
axes[1].axis('off')

plt.tight_layout()
plt.show()